In [1]:
# Import libraries and load dataset

import pandas as pd
import numpy as np
import re

df = pd.read_csv("C:/Users/VISHAL/OneDrive/Desktop/NOVITECH DA course/PYTHON 2 Project/Python phase 2/Phase 2/Uncleaned_Dataset.csv")
print("Dataset loaded successfully!")
print("Total rows:", len(df))
print("Total columns:", len(df.columns))

df.head()

Dataset loaded successfully!
Total rows: 4000
Total columns: 15


,Transaction_ID,Order_Date,Product_ID,Product_Name,Category,Unit_Price_EUR,Quantity_Sold,Discount_Percentage,Original_Price_EUR,Total_Revenue_EUR,Cost_Price_EUR,Customer_Segment,Region,Competitor_Price_EUR,Promotion_Type
0,TX00001,1/26/2024,P004,Rice 5kg,Grocery,7,1,15,8.24,7.00,6.23,Regular,Limerick,9.53,Loyalty Offer
1,TX00002,5-Apr-24,P001,Milk 1L,Dairy,1.48,1,0,1.48,1.48,0.89,Premium,Kildare,1.53,No Promotion
2,TX00003,7-Jan-24,P009,Cheddar Cheese 200g,Dairy,2.39,3,15,2.81,7.16,1.77,New Customer,Dublin,3.36,Seasonal Offer
3,TX00004,1/2/2025,P004,Rice 5kg,Grocery,7.08,6,20,8.85,42.46,6.13,Premium,Dublin,9.71,Weekend Discount
4,TX00005,12/22/2025,P007,Coffee 200g,Grocery,6.05,4,0,6.05,20.64,4.02,Regular,Galway,6.39,No Promotion


In [2]:
# Step 1: Remove Duplicate Transaction IDs

print("Duplicates before cleaning:", df["Transaction_ID"].duplicated().sum())

df = df.drop_duplicates(subset=["Transaction_ID"])

print("Step 1 completed!")
print("Rows remaining:", len(df))
print("Duplicates after cleaning:", df["Transaction_ID"].duplicated().sum())

Duplicates before cleaning: 44
Step 1 completed!
Rows remaining: 3956
Duplicates after cleaning: 0


In [3]:
# Step 2: Clean IDs

# Create product lookup from product names
product_lookup = (
    df.dropna(subset=["Product_ID", "Product_Name"])
    .assign(Product_ID=lambda x: x["Product_ID"].astype(str).str.upper())
    .groupby("Product_Name")["Product_ID"]
    .agg(lambda x: x.mode().iloc[0])
    .to_dict()
)

# Standardise Product_ID
df["Product_ID"] = df["Product_ID"].astype("string").str.strip().str.upper()

# Correct missing and wrong IDs using Product_Name
df["Product_ID"] = df.apply(
    lambda row: product_lookup.get(row["Product_Name"], row["Product_ID"])
    if pd.isna(row["Product_ID"]) or row["Product_ID"] in ["UNKNOWN", "P-001"]
    else row["Product_ID"],
    axis=1
)

# Remove rows where Product_ID is still missing
df = df.dropna(subset=["Product_ID"])

# Fix missing Transaction_ID
missing_tx = df["Transaction_ID"].isna()
df.loc[missing_tx, "Transaction_ID"] = [
    f"TX_NEW_{i:04d}" for i in range(1, missing_tx.sum() + 1)
]

# Make duplicate Transaction_ID values unique
df["Transaction_ID"] = df["Transaction_ID"].astype(str).str.strip()

duplicate_tx = df["Transaction_ID"].duplicated()
df.loc[duplicate_tx, "Transaction_ID"] = [
    f"{value}_DUP{i}"
    for i, value in enumerate(df.loc[duplicate_tx, "Transaction_ID"], 1)
]

print("Step 2 completed!")
print("Missing Product IDs:", df["Product_ID"].isna().sum())
print("Missing Transaction IDs:", df["Transaction_ID"].isna().sum())


Step 2 completed!
Missing Product IDs: 0
Missing Transaction IDs: 0


In [4]:
# Step 3: Fill Missing Product Name and Category

lookup = (
    df.dropna(subset=["Product_Name", "Category"])
    .drop_duplicates(subset="Product_ID")
)

name_map = dict(zip(lookup["Product_ID"], lookup["Product_Name"]))
category_map = dict(zip(lookup["Product_ID"], lookup["Category"]))

df["Product_Name"] = df["Product_Name"].fillna(
    df["Product_ID"].map(name_map)
)

df["Category"] = df["Category"].fillna(
    df["Product_ID"].map(category_map)
)

df = df.dropna(subset=["Product_Name", "Category"])

print("Step 3 completed!")
print("Missing Product Names:", df["Product_Name"].isna().sum())
print("Missing Categories:", df["Category"].isna().sum())
print("Rows remaining:", len(df))


Step 3 completed!
Missing Product Names: 0
Missing Categories: 0
Rows remaining: 3956


In [5]:
# Step 4: Clean Text, Customer Segment and Region

# Remove extra spaces
text_cols = ["Product_Name", "Category", "Customer_Segment", "Region"]

for col in text_cols:
    df[col] = df[col].astype("string").str.strip()

# Standardise category names
df["Category"] = df["Category"].replace({
    "GROCERY": "Grocery",
    "dairy": "Dairy",
    "DAIRY": "Dairy",
    "Fruit and Veg": "Fruit & Veg",
    "Personal care": "Personal Care"
})

# Standardise customer segments
df["Customer_Segment"] = df["Customer_Segment"].str.title()

# Standardise regions
df["Region"] = df["Region"].str.title()

# Fill missing values with most common value
df["Customer_Segment"] = df["Customer_Segment"].fillna(
    df["Customer_Segment"].mode()[0]
)

df["Region"] = df["Region"].fillna(
    df["Region"].mode()[0]
)

print("Step 4 completed!")
print("Customer Segments:", df["Customer_Segment"].unique())
print("Regions:", df["Region"].unique())


Step 4 completed!
Customer Segments: <StringArray>
['Regular', 'Premium', 'New Customer', 'Loyalty']
Length: 4, dtype: string
Regions: <StringArray>
['Limerick', 'Kildare', 'Dublin', 'Galway', 'Cork']
Length: 5, dtype: string


In [6]:

# Step 5: Clean Promotion Names

df["Promotion_Type"] = (
    df["Promotion_Type"].astype("string").str.strip().str.lower()
)

df["Promotion_Type"] = df["Promotion_Type"].replace({
    "no promotion": "No Promotion",
    "loyalty offer": "Loyalty Offer",
    "seasonal offer": "Seasonal Offer",
    "weekend discount": "Weekend Discount",
    "multi buy": "Multi-buy",
    "multi-buy": "Multi-buy"
})

# Fill missing promotion types
df["Promotion_Type"] = df["Promotion_Type"].fillna("No Promotion")

print("Step 5 completed!")
print("Promotion Types:")
print(df["Promotion_Type"].value_counts())


Step 5 completed!
Promotion Types:
No Promotion        1667
Loyalty Offer        623
Weekend Discount     564
Seasonal Offer       559
Multi-buy            543
Name: Promotion_Type, dtype: Int64


In [7]:

# Step 6: Standardise Date Format

def clean_date(x):
    if pd.isna(x) or str(x).strip() == "":
        return pd.NaT

    x = str(x).strip()

    # Handle dates with slashes
    if "/" in x:
        parts = x.split("/")

        if len(parts) == 3:
            first = int(parts[0])

            if first > 12:
                return pd.to_datetime(
                    x, format="%d/%m/%Y", errors="coerce"
                )
            else:
                return pd.to_datetime(
                    x, format="%m/%d/%Y", errors="coerce"
                )

    # Handle dates such as 5-Apr-24
    return pd.to_datetime(x, format="%d-%b-%y", errors="coerce")


df["Order_Date"] = df["Order_Date"].apply(clean_date)

# Remove only invalid or missing dates
df = df.dropna(subset=["Order_Date"])

print("Step 6 completed!")
print("Rows remaining:", len(df))
print(df["Order_Date"].head(10))


Step 6 completed!
Rows remaining: 3949
0   2024-01-26
1   2024-04-05
2   2024-01-07
3   2025-01-02
4   2025-12-22
5   2025-12-10
6   2025-04-18
7   2025-02-14
8   2024-09-28
9   2025-06-05
Name: Order_Date, dtype: datetime64[ns]


In [8]:
# Step 7: Clean Price Columns

price_cols = [
    "Unit_Price_EUR",
    "Original_Price_EUR",
    "Cost_Price_EUR",
    "Competitor_Price_EUR"
]

for col in price_cols:

    df[col] = (
        df[col].astype("string")
        .str.replace("€", "", regex=False)
        .str.replace(" ", "", regex=False)
        .str.replace(",", ".", regex=False)
    )

    df[col] = pd.to_numeric(df[col], errors="coerce")

    # Remove impossible prices
    df.loc[df[col] <= 0, col] = np.nan
    df.loc[df[col] > 100, col] = np.nan

print("Step 7 completed!")

print(df[price_cols].head(10))
print("\nMissing prices:")
print(df[price_cols].isna().sum())


Step 7 completed!
   Unit_Price_EUR  Original_Price_EUR  Cost_Price_EUR  Competitor_Price_EUR
0            7.00                8.24            6.23                  9.53
1            1.48                1.48            0.89                  1.53
2            2.39                2.81            1.77                  3.36
3            7.08                8.85            6.13                  9.71
4            6.05                6.05            4.02                  6.39
5            0.84                1.05            0.49                  1.17
6            2.91                2.91            1.56                  2.77
7            2.31                2.31            1.41                  2.32
8           10.58               10.58            7.41                  9.86
9           10.07               10.60            6.90                  9.14

Missing prices:
Unit_Price_EUR          23
Original_Price_EUR       0
Cost_Price_EUR           0
Competitor_Price_EUR     6
dtype: int64


In [9]:
# Step 8: Clean Quantity Sold

df["Quantity_Sold"] = (
    df["Quantity_Sold"].astype("string")
    .str.extract(r"(-?\d+)", expand=False)
)

df["Quantity_Sold"] = pd.to_numeric(
    df["Quantity_Sold"], errors="coerce"
)

# Replace zero and negative quantities with missing values
df.loc[df["Quantity_Sold"] <= 0, "Quantity_Sold"] = np.nan

print("Step 8 completed!")
print("Invalid quantities:", df["Quantity_Sold"].isna().sum())
print(df["Quantity_Sold"].value_counts().head(10))


Step 8 completed!
Invalid quantities: 20
1.0     1048
2.0      904
3.0      663
4.0      459
5.0      326
6.0      213
8.0      137
12.0      66
10.0      60
7.0       24
Name: Quantity_Sold, dtype: int64


In [10]:

# Step 9: Clean Discount Percentage

df["Discount_Percentage"] = (
    df["Discount_Percentage"].astype("string")
    .str.replace("%", "", regex=False)
    .str.strip()
)

df["Discount_Percentage"] = pd.to_numeric(
    df["Discount_Percentage"], errors="coerce"
)

# Remove impossible discount values
df.loc[
    (df["Discount_Percentage"] < 0) |
    (df["Discount_Percentage"] > 100),
    "Discount_Percentage"
] = np.nan

print("Step 9 completed!")
print("Invalid discounts:", df["Discount_Percentage"].isna().sum())
print(df["Discount_Percentage"].value_counts().sort_index())


Step 9 completed!
Invalid discounts: 15
0.0     1659
5.0      374
10.0     679
15.0     501
20.0     371
25.0     252
30.0      98
Name: Discount_Percentage, dtype: int64


In [11]:

# Step 10: Fill Missing Prices and Discounts

fill_cols = [
    "Unit_Price_EUR",
    "Original_Price_EUR",
    "Cost_Price_EUR",
    "Competitor_Price_EUR",
    "Quantity_Sold",
    "Discount_Percentage"
]

for col in fill_cols:
    df[col] = df[col].fillna(df[col].median())

# Quantity must be a whole number
df["Quantity_Sold"] = df["Quantity_Sold"].round().astype(int)

# Discount should be a whole percentage
df["Discount_Percentage"] = df["Discount_Percentage"].round(2)

print("Step 10 completed!")
print("Missing values after filling:")
print(df[fill_cols].isna().sum())


Step 10 completed!
Missing values after filling:
Unit_Price_EUR          0
Original_Price_EUR      0
Cost_Price_EUR          0
Competitor_Price_EUR    0
Quantity_Sold           0
Discount_Percentage     0
dtype: int64


In [12]:

# Step 11: Correct Total Revenue

df["Total_Revenue_EUR"] = pd.to_numeric(
    df["Total_Revenue_EUR"], errors="coerce"
)

# Calculate correct revenue
calculated_revenue = (
    df["Unit_Price_EUR"] * df["Quantity_Sold"]
)

# Replace missing or incorrect revenue
wrong_revenue = (
    df["Total_Revenue_EUR"].isna() |
    ~np.isclose(
        df["Total_Revenue_EUR"],
        calculated_revenue,
        atol=0.02
    )
)

df.loc[wrong_revenue, "Total_Revenue_EUR"] = (
    calculated_revenue[wrong_revenue]
)

df["Total_Revenue_EUR"] = df["Total_Revenue_EUR"].round(2)

print("Step 11 completed!")
print("Revenue corrections:", wrong_revenue.sum())
print(df[["Unit_Price_EUR", "Quantity_Sold",
          "Total_Revenue_EUR"]].head(10))


Step 11 completed!
Revenue corrections: 264
   Unit_Price_EUR  Quantity_Sold  Total_Revenue_EUR
0            7.00              1               7.00
1            1.48              1               1.48
2            2.39              3               7.16
3            7.08              6              42.46
4            6.05              4              24.20
5            0.84              3               2.51
6            2.91              3               8.72
7            2.31              8              18.47
8           10.58              2              21.17
9           10.07              5              50.37


In [13]:

# Step 12: Add Profit Columns

# Calculate total cost
df["Total_Cost_EUR"] = (
    df["Cost_Price_EUR"] * df["Quantity_Sold"]
)

# Calculate profit
df["Profit_EUR"] = (
    df["Total_Revenue_EUR"] - df["Total_Cost_EUR"]
)

# Calculate profit margin
df["Profit_Margin_Percentage"] = (
    df["Profit_EUR"] / df["Total_Revenue_EUR"] * 100
)

# Round calculated values
df["Total_Cost_EUR"] = df["Total_Cost_EUR"].round(2)
df["Profit_EUR"] = df["Profit_EUR"].round(2)
df["Profit_Margin_Percentage"] = (
    df["Profit_Margin_Percentage"].round(2)
)

print("Step 12 completed!")
print(df[[
    "Total_Revenue_EUR",
    "Total_Cost_EUR",
    "Profit_EUR",
    "Profit_Margin_Percentage"
]].head(10))


Step 12 completed!
   Total_Revenue_EUR  Total_Cost_EUR  Profit_EUR  Profit_Margin_Percentage
0               7.00            6.23        0.77                     11.00
1               1.48            0.89        0.59                     39.86
2               7.16            5.31        1.85                     25.84
3              42.46           36.78        5.68                     13.38
4              24.20           16.08        8.12                     33.55
5               2.51            1.47        1.04                     41.43
6               8.72            4.68        4.04                     46.33
7              18.47           11.28        7.19                     38.93
8              21.17           14.82        6.35                     30.00
9              50.37           34.50       15.87                     31.51


In [14]:

# Final Step: Check and Export Cleaned Dataset

# Check missing values
print("Missing values in final dataset:")
print(df.isna().sum())

# Check duplicate records
print("\nDuplicate rows:", df.duplicated().sum())

# Convert dates to standard format
df["Order_Date"] = pd.to_datetime(
    df["Order_Date"]
).dt.strftime("%Y-%m-%d")

# Export cleaned dataset
df.to_csv(
    "Price_Sensitivity_Cleaned_Dataset.csv",
    index=False
)

print("\nCleaning completed successfully!")
print("Final rows:", len(df))
print("Final columns:", len(df.columns))
print("CSV file saved successfully!")

Missing values in final dataset:
Transaction_ID              0
Order_Date                  0
Product_ID                  0
Product_Name                0
Category                    0
Unit_Price_EUR              0
Quantity_Sold               0
Discount_Percentage         0
Original_Price_EUR          0
Total_Revenue_EUR           0
Cost_Price_EUR              0
Customer_Segment            0
Region                      0
Competitor_Price_EUR        0
Promotion_Type              0
Total_Cost_EUR              0
Profit_EUR                  0
Profit_Margin_Percentage    0
dtype: int64

Duplicate rows: 0

Cleaning completed successfully!
Final rows: 3949
Final columns: 18
CSV file saved successfully!
